# GameTheory-06i — Ensembles limites : Poincaré-Bendixson en dimension 2

**Navigation** : [<< 6-EvolutionTrust](GameTheory-06-EvolutionTrust-Python.ipynb) | [Version Python](GameTheory-06i-Ensembles-Limites-Poincare-Bendixson-Python.ipynb) | [Index](README.md)

Où peut bien finir une trajectoire d'apprentissage ? Posée dans le plan, la question a une réponse fermée : le théorème de **Poincaré-Bendixson** dit qu'un champ de vecteurs planaire ne peut faire qu'une de trois choses à une trajectoire — la faire converger vers un point, la faire tourner en boucle fermée, ou la faire coller à un cycle de coins. Pas de quatrième issue, et en particulier : **pas de chaos**.

Czechowski et Piliouras (2021) ont montré que ce théorème s'applique à la dynamique de replicateur dans les deux cadres planaires de la théorie des jeux : une population sur trois stratégies, et deux populations sur un jeu 2×2. Ce notebook **exécute** les trois issues — chacune fabriquée par un jeu canonique, intégrée par RK4, puis classée par un détecteur mécanique qui ne lit que la trajectoire.

Toute la mécanique est portée en C# depuis le module compagnon [`limit_sets.py`](limit_sets.py) (deux cadres, intégrateur RK4, détecteur de régime), validé par [`tests/test_limit_sets.py`](tests/test_limit_sets.py) — 16 tests, dont la stabilité du verdict au pas d'intégration (dt vs dt/2) et à la condition initiale dans le bassin.

**Plan** : le théorème énoncé (section 1) ; les trois issues exécutées (section 2) ; le mur $w = l$ de la famille Pierre-Feuille-Ciseaux relié au vocabulaire chambres/murs du [GameTheory-3b](GameTheory-03b-Chambres-Murs-Chemins-Lean-Python.ipynb) (section 3) ; trois exercices (section 4) ; et pour finir, la raison pour laquelle tout ceci **tombe** dès qu'on quitte le plan.

In [1]:
// Port BCL des fonctions utilisées de limit_sets.py : champs, RK4, détecteur et jeux.
using System;
using System.Linq;
using System.Collections.Generic;
using System.Globalization;
using System.Text;
using Microsoft.DotNet.Interactive.Formatting;

CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentUICulture = CultureInfo.InvariantCulture;

public class MesuresRegime
{
    public int Segments;
    public double[] Sejours = Array.Empty<double>();
    public double Drift;
    public double TempsRetour = double.NaN;
    public double DistanceRetour = double.NaN;
    public double DistanceFrontiere = double.NaN;
}

public static class Limites
{
    public static (double[,], double[,]) Prisonnier() =>
        (new double[,] {{3, 0}, {5, 1}}, new double[,] {{3, 5}, {0, 1}});
    public static (double[,], double[,]) MatchingPennies() =>
        (new double[,] {{1, -1}, {-1, 1}}, new double[,] {{-1, 1}, {1, -1}});
    public static (double[,], double[,]) ChasseAuCerf() =>
        (new double[,] {{4, 1}, {3, 2}}, new double[,] {{4, 3}, {1, 2}});
    public static double[,] Rps(double w, double l) =>
        new double[,] {{0, -l, w}, {w, 0, -l}, {-l, w, 0}};
    public static double[] Etat(double x, double y) => new[] {x, 1 - x, y, 1 - y};
    public static (double, double) LireEtat(double[] z) => (z[0], z[2]);
    public static double InvariantMp(double[] z) => z[0] * (1-z[0]) * z[2] * (1-z[2]);

    public static double[] Champ1(double[,] a, double[] x)
    {
        int n = x.Length;
        var f = new double[n];
        for (int i = 0; i < n; i++)
            for (int j = 0; j < n; j++) f[i] += a[i,j] * x[j];
        double moyenne = 0;
        for (int i = 0; i < n; i++) moyenne += x[i] * f[i];
        return Enumerable.Range(0,n).Select(i => x[i] * (f[i] - moyenne)).ToArray();
    }

    public static double[] Champ2(double[,] a, double[,] b, double[] z)
    {
        double f0 = a[0,0]*z[2] + a[0,1]*z[3];
        double f1 = a[1,0]*z[2] + a[1,1]*z[3];
        double g0 = b[0,0]*z[0] + b[1,0]*z[1];
        double g1 = b[0,1]*z[0] + b[1,1]*z[1];
        double mx = z[0]*f0 + z[1]*f1, my = z[2]*g0 + z[3]*g1;
        return new[] {z[0]*(f0-mx), z[1]*(f1-mx), z[2]*(g0-my), z[3]*(g1-my)};
    }

    public static (double[] temps, double[][] trajet) Integrer(Func<double[],double[]> champ, double[] initial, double tmax, double dt)
    {
        int n = (int)Math.Round(tmax / dt);
        var temps = new double[n+1];
        var trajet = new double[n+1][];
        var z = (double[])initial.Clone();
        trajet[0] = (double[])z.Clone();
        for (int k = 0; k < n; k++)
        {
            var k1 = champ(z);
            var k2 = champ(Enumerable.Range(0,z.Length).Select(i => z[i] + 0.5*dt*k1[i]).ToArray());
            var k3 = champ(Enumerable.Range(0,z.Length).Select(i => z[i] + 0.5*dt*k2[i]).ToArray());
            var k4 = champ(Enumerable.Range(0,z.Length).Select(i => z[i] + dt*k3[i]).ToArray());
            z = Enumerable.Range(0,z.Length).Select(i => z[i] + (dt/6.0)*(k1[i]+2*k2[i]+2*k3[i]+k4[i])).ToArray();
            temps[k+1] = (k+1)*dt;
            trajet[k+1] = z;
        }
        return (temps, trajet);
    }

    public static double Mediane(IEnumerable<double> valeurs)
    {
        var v = valeurs.OrderBy(x => x).ToArray();
        if (v.Length == 0) throw new ArgumentException("Médiane vide");
        return v.Length % 2 == 1 ? v[v.Length/2] : (v[v.Length/2-1]+v[v.Length/2])/2.0;
    }

    public static (List<int[]> dominants, List<double> bornes) Episodes(double[] temps, double[][] trajet, double share = 0.5)
    {
        int dim = trajet[0].Length;
        var blocs = dim == 4 ? new[] {(0,2), (2,4)} : new[] {(0,dim)};
        Func<double[],int[]> etiquette = z => blocs.Select(bloc => {
            int meilleur = bloc.Item1;
            for (int i = bloc.Item1+1; i < bloc.Item2; i++) if (z[i] > z[meilleur]) meilleur = i;
            return meilleur-bloc.Item1;
        }).ToArray();
        var dominants = new List<int[]> {etiquette(trajet[0])};
        var bornes = new List<double> {0.0};
        for (int i = 0; i < temps.Length; i++)
        {
            var lab = etiquette(trajet[i]);
            var precedent = dominants[dominants.Count-1];
            var changes = Enumerable.Range(0,blocs.Length).Where(p => lab[p] != precedent[p]).ToArray();
            if (changes.Length > 0 && changes.All(p => trajet[i][blocs[p].Item1+lab[p]] > share))
            {
                dominants.Add(lab);
                bornes.Add(temps[i]);
            }
        }
        return (dominants,bornes);
    }

    public static double Distance(double[] a, double[] b)
    {
        double somme = 0;
        for (int j = 0; j < a.Length; j++) { double d = a[j]-b[j]; somme += d*d; }
        return Math.Sqrt(somme);
    }

    public static (int index, double distance) PremierRetour(double[] temps, double[][] trajet, double tolerance, double tSkip)
    {
        int debut = Array.BinarySearch(temps, tSkip);
        if (debut < 0) debut = ~debut;
        for (int i = debut; i < temps.Length; i++)
        {
            double d = Distance(trajet[i], trajet[0]);
            if (d <= tolerance) return (i,d);
        }
        return (-1,double.NaN);
    }

    public static (string verdict, MesuresRegime info) Detecter(double[] temps, double[][] trajet,
        double tolPoint = 2e-3, double tolRetour = 5e-2, double tolBord = 1e-3,
        double fraction = 0.2, double escapeGrowth = 1.5, double escapeFloor = 1.0)
    {
        int n = temps.Length, w = Math.Max(2,(int)(fraction*temps.Length));
        var episodes = Episodes(temps,trajet);
        var gaps = Enumerable.Range(1,episodes.bornes.Count-1)
            .Select(i => episodes.bornes[i]-episodes.bornes[i-1]).ToArray();
        bool escaped = gaps.Length >= 3 && gaps[gaps.Length-1] >= escapeFloor &&
            gaps[gaps.Length-1] >= escapeGrowth*Mediane(gaps.Take(gaps.Length-1));
        var info = new MesuresRegime {Segments = episodes.dominants.Count, Sejours = gaps};
        double drift = 0;
        for (int i = n-w; i < n; i++) drift = Math.Max(drift,Distance(trajet[i],trajet[n-1]));
        info.Drift = drift;
        if (drift < tolPoint && !escaped) return ("point_fixe",info);
        var retour = PremierRetour(temps,trajet,tolRetour,temps[w]);
        if (retour.index >= 0)
        {
            info.TempsRetour = temps[retour.index];
            info.DistanceRetour = retour.distance;
            return ("orbite_periodique",info);
        }
        double minimum = double.PositiveInfinity;
        for (int i = n-w; i < n; i++) foreach (double x in trajet[i]) minimum = Math.Min(minimum,x);
        info.DistanceFrontiere = minimum;
        if (minimum < tolBord || escaped) return ("cycle_heteroclinique",info);
        return ("indetermine",info);
    }

    public static void Verifier(string nom, double valeur, double reference, double tolerance)
    {
        double ecart = Math.Abs(valeur-reference);
        Console.WriteLine($"{nom} : calculé {valeur:E6}, référence Python CITÉE {reference:E6}, écart {ecart:E3}, tolérance {tolerance:E3}");
        if (double.IsNaN(valeur) || ecart > tolerance) throw new Exception($"Écart : {nom}");
    }

    public static string SvgCourbe(double[][] trajet, int ix, int iy, string titre, string xlabel, string ylabel, string couleur = "#1f77b4")
    {
        var s = new StringBuilder();
        s.Append("<svg xmlns='http://www.w3.org/2000/svg' width='650' height='600' viewBox='0 0 650 600'>");
        s.Append("<rect width='650' height='600' fill='white'/><rect x='65' y='45' width='520' height='490' fill='none' stroke='black'/>");
        s.Append("<text x='325' y='24' text-anchor='middle' font-size='16'>" + System.Net.WebUtility.HtmlEncode(titre) + "</text>");
        s.Append("<text x='325' y='575' text-anchor='middle'>" + System.Net.WebUtility.HtmlEncode(xlabel) + "</text>");
        s.Append("<text x='15' y='290' transform='rotate(-90 15 290)' text-anchor='middle'>" + System.Net.WebUtility.HtmlEncode(ylabel) + "</text>");
        s.Append("<polyline fill='none' stroke='" + couleur + "' stroke-width='1.2' points='");
        int pas = Math.Max(1,trajet.Length/3000);
        for (int i = 0; i < trajet.Length; i += pas)
            s.AppendFormat(CultureInfo.InvariantCulture,"{0:F2},{1:F2} ",65+520*trajet[i][ix],535-490*trajet[i][iy]);
        var fin = trajet[trajet.Length-1];
        s.AppendFormat(CultureInfo.InvariantCulture,"{0:F2},{1:F2}'/>",65+520*fin[ix],535-490*fin[iy]);
        s.AppendFormat(CultureInfo.InvariantCulture,"<circle cx='{0:F2}' cy='{1:F2}' r='5' fill='green'/>",65+520*trajet[0][ix],535-490*trajet[0][iy]);
        s.AppendFormat(CultureInfo.InvariantCulture,"<circle cx='{0:F2}' cy='{1:F2}' r='5' fill='red'/>",65+520*fin[ix],535-490*fin[iy]);
        s.Append("<text x='75' y='65'>départ : vert ; arrivée : rouge</text></svg>");
        return s.ToString();
    }
}

// Contrôle sanitaire du module : RK4 conserve exactement les sommes des
// populations (invariant lineaire du champ) sur un court run Prisonnier.
var (aSanitaire,bSanitaire) = Limites.Prisonnier();
var (tempsSanitaire,trajetSanitaire) = Limites.Integrer(z => Limites.Champ2(aSanitaire,bSanitaire,z), Limites.Etat(0.6,0.4), 2.0, 1e-3);
double errSommes = trajetSanitaire.Max(z => Math.Max(Math.Abs(z[0]+z[1]-1),Math.Abs(z[2]+z[3]-1)));
Console.WriteLine($"Imports OK - conservation RK4 des sommes : erreur max {errSommes:E1}");
Limites.Verifier("Conservation RK4",errSommes,1.6e-15,2e-15);

Imports OK - conservation RK4 des sommes : erreur max 1.6E-015


Conservation RK4 : calculé 1.554312E-015, référence Python CITÉE 1.600000E-015, écart 4.569E-017, tolérance 2.000E-015


## 1. Le théorème : trois issues, pas une de plus

**Poincaré-Bendixson (forme utilisée).** Soit un champ de vecteurs continu sur une région fermée bornée du plan, avec un nombre fini de points fixes. L'ensemble $\omega$-limite d'une trajectoire qui reste confinée dans cette région est exactement l'un des trois objets suivants :

1. **un point fixe** — la trajectoire converge ;
2. **une orbite périodique** — la trajectoire parcourt une boucle fermée ;
3. **un cycle hétéroclinique** — une concaténation de trajectoires reliant des points fixes (typiquement des coins du domaine), parcourue sans jamais terminer.

Le théorème n'est pas prouvé ici : il est *exécuté*. Chaque issue est fabriquée par un jeu canonique, intégrée numériquement, et classée par le détecteur `Detecter` — qui ne connaît pas le jeu, seulement la trajectoire. Ses trois signatures sont mécaniques : immobilité de la fenêtre terminale (point fixe), premier retour au point de départ (orbite périodique), collage à la frontière avec allongement des séjours (cycle hétéroclinique).

Pourquoi le théorème s'applique : la dynamique de replicateur $\dot{x}_i = x_i\,((Ax)_i - x^\top A x)$ vit dans le simplexe, et nos deux cadres sont planaires — le simplexe d'une population sur trois stratégies est un **triangle**, le produit des simplexes de deux populations sur un jeu 2×2 est un **carré**. Dans les deux cas, la dimension de l'espace d'états est 2.

In [2]:
// Deux contrôles du cadre : les centres sont bien des points fixes du champ.
// Les chaînes SVG sont transmises au moteur HTML sans échappement pour rendre les figures.
Formatter.Register<string>((value, writer) => writer.Write(value), "text/html");
var (aMpCentre,bMpCentre) = Limites.MatchingPennies();
var centreMp = Limites.Etat(0.5,0.5);
var champMpCentre = Limites.Champ2(aMpCentre,bMpCentre,centreMp);
var barycentre = new[] {1.0/3,1.0/3,1.0/3};
var champRpsCentre = Limites.Champ1(Limites.Rps(1.5,1.5),barycentre);
Console.WriteLine("Matching Pennies, champ au centre du carre : [" + string.Join(" ",champMpCentre.Select(x => Math.Round(x,12).ToString("0.############"))) + "]");
Console.WriteLine("RPS mur w=l, champ au barycentre          : [" + string.Join(" ",champRpsCentre.Select(x => Math.Round(x,12).ToString("0.############"))) + "]");
foreach (var x in champMpCentre) Limites.Verifier("Centre MP",x,0,1e-12);
foreach (var x in champRpsCentre) Limites.Verifier("Barycentre RPS",x,0,1e-12);

Matching Pennies, champ au centre du carre : [0 0 0 0]


RPS mur w=l, champ au barycentre          : [0 0 0]


Centre MP : calculé 0,000000E+000, référence Python CITÉE 0,000000E+000, écart 0,000E+000, tolérance 1,000E-012


Centre MP : calculé 0,000000E+000, référence Python CITÉE 0,000000E+000, écart 0,000E+000, tolérance 1,000E-012


Centre MP : calculé 0,000000E+000, référence Python CITÉE 0,000000E+000, écart 0,000E+000, tolérance 1,000E-012


Centre MP : calculé 0,000000E+000, référence Python CITÉE 0,000000E+000, écart 0,000E+000, tolérance 1,000E-012


Barycentre RPS : calculé 0,000000E+000, référence Python CITÉE 0,000000E+000, écart 0,000E+000, tolérance 1,000E-012


Barycentre RPS : calculé 0,000000E+000, référence Python CITÉE 0,000000E+000, écart 0,000E+000, tolérance 1,000E-012


Barycentre RPS : calculé 0,000000E+000, référence Python CITÉE 0,000000E+000, écart 0,000E+000, tolérance 1,000E-012


### Lecture des contrôles

Les deux champs s'annulent aux centres — le centre du carré (Matching Pennies) et le barycentre (RPS sur le mur) sont des points fixes. Ce seront justement les points autour desquels les deux régimes non triviaux s'organisent : un **centre** entouré d'orbites fermées dans un cas, un point dont la stabilité change au mur $w = l$ dans l'autre.

## 2. Trois trajectoires, trois issues

### 2.1 Dilemme du Prisonnier — l'issue point fixe

Deux populations, actions $(C, D)$, gains $T=5, R=3, P=1, S=0$. La trahison domine strictement : l'équilibre $(D, D)$ est un coin du carré. La trajectoire partant de $(0.6, 0.4)$ (60 % de coopérateurs chez la population 1, 40 % chez la population 2) doit y converger.

In [3]:
// Intégration RK4 à pas fixe et classification par le détecteur porté.
var (aPd,bPd) = Limites.Prisonnier();
var (tPd,trPd) = Limites.Integrer(z => Limites.Champ2(aPd,bPd,z),Limites.Etat(0.6,0.4),30.0,1e-3);
var (regimePd,infoPd) = Limites.Detecter(tPd,trPd);
var (xPd,yPd) = Limites.LireEtat(trPd[trPd.Length-1]);
Console.WriteLine($"verdict       : {regimePd}");
Console.WriteLine($"etat final    : x = {xPd:E2} (part de C), y = {yPd:E2} (part de C)");
Console.WriteLine($"drift terminal: {infoPd.Drift:E2}");
if (regimePd != "point_fixe") throw new Exception("Verdict différent de la référence Python CITÉE");
Limites.Verifier("État final x",xPd,9.66e-14,0.005e-14);
Limites.Verifier("État final y",yPd,2.86e-14,0.005e-14);
Limites.Verifier("Drift terminal",infoPd.Drift,5.73e-11,0.005e-11);
// Portrait calculé depuis la trajectoire, départ en vert et arrivée en rouge.
display(Limites.SvgCourbe(trPd,0,2,"Dilemme du Prisonnier : convergence au coin (D, D)","x : part de cooperer, population 1","y : part de cooperer, population 2"), "text/html");

verdict       : point_fixe


etat final    : x = 9,66E-014 (part de C), y = 2,86E-014 (part de C)


drift terminal: 5,73E-011


État final x : calculé 9,660537E-014, référence Python CITÉE 9,660000E-014, écart 5,367E-018, tolérance 5,000E-017


État final y : calculé 2,862381E-014, référence Python CITÉE 2,860000E-014, écart 2,381E-017, tolérance 5,000E-017


Drift terminal : calculé 5,727522E-011, référence Python CITÉE 5,730000E-011, écart 2,478E-014, tolérance 5,000E-014


Dilemme du Prisonnier : convergence au coin (D, D) x : part de cooperer, population 1 y : part de cooperer, population 2 <polyline fill='none' stroke='#1f77b4' stroke-width='1.2' points='377.00,339.00 375.25,340.88 373.51,342.74 371.76,344.60 370.01,346.44 368.27,348.28 366.52,350.10 364.78,351.92 363.03,353.72 361.29,355.51 359.55,357.29 357.81,359.06 356.07,360.82 354.34,362.56 352.60,364.29 350.87,366.02 349.14,367.73 347.41,369.42 345.69,371.11 343.97,372.78 342.25,374.44 340.53,376.09 338.81,377.73 337.10,379.35 335.40,380.96 333.69,382.56 331.99,384.15 330.29,385.72 328.60,387.29 326.91,388.83 325.23,390.37 323.54,391.89 321.87,393.40 320.19,394.90 318.52,396.39 316.86,397.86 315.20,399.32 313.54,400.77 311.89,402.20 310.25,403.62 308.61,405.03 306.97,406.43 305.34,407.81 303.72,409.18 302.10,410.54 300.48,411.88 298.87,413.22 297.27,414.54 295.67,415.84 294.08,417.14 292.49,418.42 290.91,419.69 289.33,420.95 287.76,422.20 286.20,423.43 284.64,424.66 283.09,425.87 281.55,427.06 280.01,428.25 278.47,429.42 276.95,430.59 275.43,431.74 273.91,432.88 272.41,434.00 270.91,435.12 269.41,436.22 267.93,437.32 266.44,438.40 264.97,439.47 263.50,440.53 262.04,441.58 260.59,442.61 259.14,443.64 257.70,444.65 256.27,445.66 254.84,446.65 253.42,447.64 252.01,448.61 250.61,449.57 249.21,450.52 247.82,451.47 246.43,452.40 245.06,453.32 243.69,454.23 242.32,455.13 240.97,456.02 239.62,456.90 238.28,457.78 236.95,458.64 235.62,459.49 234.30,460.34 232.99,461.17 231.68,461.99 230.38,462.81 229.09,463.62 227.81,464.42 226.54,465.20 225.27,465.98 224.01,466.76 222.75,467.52 221.51,468.27 220.27,469.02 219.03,469.76 217.81,470.48 216.59,471.21 215.38,471.92 214.18,472.62 212.98,473.32 211.79,474.01 210.61,474.69 209.44,475.36 208.27,476.03 207.11,476.68 205.96,477.34 204.82,477.98 203.68,478.61 202.55,479.24 201.43,479.86 200.31,480.48 199.20,481.09 198.10,481.69 197.00,482.28 195.92,482.87 194.84,483.44 193.76,484.02 192.70,484.58 191.64,485.14 190.59,485.70 189.54,486.25 188.50,486.79 187.47,487.32 186.45,487.85 185.43,488.37 184.42,488.89 183.42,489.40 182.42,489.91 181.43,490.40 180.45,490.90 179.48,491.39 178.51,491.87 177.55,492.34 176.59,492.81 175.64,493.28 174.70,493.74 173.76,494.20 172.84,494.64 171.91,495.09 171.00,495.53 170.09,495.96 169.19,496.39 168.29,496.82 167.40,497.24 166.52,497.65 165.65,498.06 164.78,498.47 163.91,498.87 163.06,499.26 162.21,499.66 161.36,500.04 160.52,500.42 159.69,500.80 158.87,501.18 158.05,501.55 157.23,501.91 156.43,502.27 155.63,502.63 154.83,502.98 154.04,503.33 153.26,503.68 152.48,504.02 151.71,504.35 150.95,504.69 150.19,505.02 149.44,505.34 148.69,505.67 147.95,505.98 147.21,506.30 146.48,506.61 145.76,506.92 145.04,507.22 144.33,507.52 143.62,507.82 142.92,508.11 142.22,508.40 141.53,508.69 140.85,508.97 140.17,509.26 139.49,509.53 138.82,509.81 138.16,510.08 137.50,510.35 136.85,510.61 136.20,510.87 135.56,511.13 134.92,511.39 134.29,511.64 133.67,511.89 133.04,512.14 132.43,512.39 131.82,512.63 131.21,512.87 130.61,513.11 130.01,513.34 129.42,513.57 128.83,513.80 128.25,514.03 127.67,514.25 127.10,514.47 126.54,514.69 125.97,514.91 125.41,515.12 124.86,515.33 124.31,515.54 123.77,515.75 123.23,515.96 122.69,516.16 122.16,516.36 121.64,516.56 121.12,516.75 120.60,516.94 120.09,517.14 119.58,517.33 119.07,517.51 118.57,517.70 118.08,517.88 117.59,518.06 117.10,518.24 116.62,518.42 116.14,518.59 115.67,518.77 115.20,518.94 114.73,519.11 114.27,519.27 113.81,519.44 113.35,519.60 112.90,519.77 112.46,519.93 112.01,520.09 111.58,520.24 111.14,520.40 110.71,520.55 110.28,520.70 109.86,520.85 109.44,521.00 109.02,521.15 108.61,521.29 108.20,521.44 107.80,521.58 107.39,521.72 107.00,521.86 106.60,522.00 106.21,522.13 105.82,522.27 105.44,522.40 105.06,522.53 104.68,522.66 104.31,522.79 103.94,522.92 103.57,523.04 103.20,523.17 102.84,523.29 102.49,523.41 102.13,523.53 101.78,523.65 101.43,523.77 101.09,523.89 100.74,524.00 100.41,524.12 100.07,524.23 99.74,524.34 99.41,524.45 99.08,524.56 98.76,52

### Lecture du résultat — l'issue point fixe du Dilemme du Prisonnier

Première issue, la plus simple : la trajectoire meurt au coin $(D, D)$ — l'équilibre strict est un attracteur, le drift terminal s'effondre (5,7·10⁻¹¹ : la fenêtre terminale est immobile). Le détecteur rend `point_fixe`. Notons ce que le théorème garantit ici : peu importe le point de départ intérieur, l'ensemble $\omega$-limite sera un objet de la liste — en l'occurrence toujours ce coin.

### 2.2 Matching Pennies — l'issue orbite périodique

$(+1, -1)$ : chaque joueur veut s'aligner sur l'autre (Pile avec Pile), l'autre veut différer. Le jeu n'a aucun équilibre pur ; le mixte $(1/2, 1/2)$ est un **centre** entouré d'orbites fermées, de niveau l'invariant $x(1-x)\,y(1-y)$.

In [4]:
// Intégration et détection avec les mêmes paramètres que le carnet Python.
var (aMp,bMp) = Limites.MatchingPennies();
Func<double[],double[]> rhsMp = z => Limites.Champ2(aMp,bMp,z);
var (tMp,trMp) = Limites.Integrer(rhsMp,Limites.Etat(0.35,0.6),20.0,1e-3);
var invMp = trMp.Select(Limites.InvariantMp).ToArray();
var (regimeMp,infoMp) = Limites.Detecter(tMp,trMp);
double conservationMp = (invMp.Max()-invMp.Min())/invMp[0];
Console.WriteLine($"verdict : {regimeMp}");
Console.WriteLine($"conservation relative de l'invariant x(1-x)y(1-y) : {conservationMp:E2}");
Console.WriteLine($"premier retour : t = {infoMp.TempsRetour:F3}, distance = {infoMp.DistanceRetour:G17}");
if (regimeMp != "orbite_periodique") throw new Exception("Verdict différent de la référence Python CITÉE");
// Les références sont les sorties citées du jumeau Python ; la conservation
// peut différer à l'échelle de l'arrondi flottant selon l'implémentation.
Limites.Verifier("Conservation relative MP",conservationMp,1.94e-14,3e-14);
Limites.Verifier("Premier retour MP : temps",infoMp.TempsRetour,6.292,1e-9);
Limites.Verifier("Premier retour MP : distance",infoMp.DistanceRetour,0.04979660782048693,1e-10);
// Portrait calculé à partir de la trajectoire intégrée.
display(Limites.SvgCourbe(trMp,0,2,"Matching Pennies : orbite fermée autour du centre","x : part de Pile, joueur 1","y : part de Pile, joueur 2","#9467bd"),"text/html");

verdict : orbite_periodique


conservation relative de l'invariant x(1-x)y(1-y) : 1,94E-014


premier retour : t = 6,292, distance = 0,049796607820486928


Conservation relative MP : calculé 1,944415E-014, référence Python CITÉE 1,940000E-014, écart 4,415E-017, tolérance 3,000E-014


Premier retour MP : temps : calculé 6,292000E+000, référence Python CITÉE 6,292000E+000, écart 0,000E+000, tolérance 1,000E-009


Premier retour MP : distance : calculé 4,979661E-002, référence Python CITÉE 4,979661E-002, écart 0,000E+000, tolérance 1,000E-010


Matching Pennies : orbite fermée autour du centre x : part de Pile, joueur 1 y : part de Pile, joueur 2 <polyline fill='none' stroke='#9467bd' stroke-width='1.2' points='247.00,241.00 247.29,240.58 247.57,240.16 247.86,239.74 248.16,239.32 248.45,238.91 248.75,238.49 249.05,238.08 249.36,237.67 249.66,237.27 249.97,236.86 250.28,236.46 250.60,236.06 250.91,235.66 251.23,235.26 251.55,234.86 251.88,234.47 252.21,234.08 252.53,233.69 252.87,233.30 253.20,232.92 253.54,232.54 253.88,232.16 254.22,231.78 254.56,231.40 254.91,231.03 255.26,230.65 255.61,230.28 255.97,229.92 256.32,229.55 256.68,229.19 257.04,228.83 257.41,228.47 257.78,228.11 258.14,227.75 258.52,227.40 258.89,227.05 259.27,226.70 259.64,226.36 260.02,226.01 260.41,225.67 260.79,225.33 261.18,225.00 261.57,224.66 261.96,224.33 262.36,224.00 262.76,223.67 263.15,223.35 263.56,223.03 263.96,222.71 264.37,222.39 264.77,222.07 265.18,221.76 265.60,221.45 266.01,221.14 266.43,220.84 266.85,220.53 267.27,220.23 267.69,219.93 268.12,219.64 268.55,219.34 268.98,219.05 269.41,218.76 269.84,218.47 270.28,218.19 270.72,217.91 271.16,217.63 271.60,217.35 272.04,217.08 272.49,216.81 272.94,216.54 273.39,216.27 273.84,216.01 274.29,215.75 274.75,215.49 275.20,215.23 275.66,214.98 276.13,214.72 276.59,214.48 277.05,214.23 277.52,213.99 277.99,213.74 278.46,213.51 278.93,213.27 279.41,213.04 279.88,212.80 280.36,212.58 280.84,212.35 281.32,212.13 281.80,211.91 282.29,211.69 282.77,211.47 283.26,211.26 283.75,211.05 284.24,210.84 284.73,210.64 285.23,210.43 285.72,210.23 286.22,210.04 286.72,209.84 287.22,209.65 287.72,209.46 288.22,209.27 288.73,209.09 289.23,208.91 289.74,208.73 290.25,208.55 290.76,208.38 291.27,208.21 291.78,208.04 292.30,207.88 292.81,207.71 293.33,207.55 293.85,207.40 294.37,207.24 294.89,207.09 295.41,206.94 295.93,206.79 296.45,206.65 296.98,206.51 297.51,206.37 298.03,206.23 298.56,206.10 299.09,205.97 299.62,205.84 300.15,205.72 300.68,205.59 301.22,205.47 301.75,205.36 302.29,205.24 302.82,205.13 303.36,205.02 303.90,204.92 304.44,204.81 304.97,204.71 305.52,204.61 306.06,204.52 306.60,204.42 307.14,204.33 307.68,204.25 308.23,204.16 308.77,204.08 309.32,204.00 309.86,203.93 310.41,203.85 310.96,203.78 311.50,203.71 312.05,203.65 312.60,203.58 313.15,203.52 313.70,203.47 314.25,203.41 314.80,203.36 315.35,203.31 315.90,203.27 316.46,203.22 317.01,203.18 317.56,203.14 318.11,203.11 318.67,203.07 319.22,203.04 319.77,203.02 320.33,202.99 320.88,202.97 321.43,202.95 321.99,202.94 322.54,202.92 323.10,202.91 323.65,202.90 324.21,202.90 324.76,202.90 325.32,202.90 325.87,202.90 326.42,202.90 326.98,202.91 327.53,202.92 328.09,202.94 328.64,202.95 329.20,202.97 329.75,203.00 330.30,203.02 330.86,203.05 331.41,203.08 331.96,203.11 332.52,203.15 333.07,203.19 333.62,203.23 334.17,203.27 334.72,203.32 335.27,203.37 335.83,203.42 336.38,203.47 336.93,203.53 337.47,203.59 338.02,203.66 338.57,203.72 339.12,203.79 339.67,203.86 340.21,203.94 340.76,204.01 341.30,204.09 341.85,204.17 342.39,204.26 342.93,204.35 343.48,204.44 344.02,204.53 344.56,204.63 345.10,204.72 345.64,204.83 346.18,204.93 346.72,205.04 347.25,205.15 347.79,205.26 348.32,205.37 348.86,205.49 349.39,205.61 349.92,205.73 350.45,205.86 350.98,205.99 351.51,206.12 352.04,206.25 352.57,206.39 353.09,206.53 353.62,206.67 354.14,206.81 354.66,206.96 355.19,207.11 355.71,207.26 356.22,207.42 356.74,207.58 357.26,207.74 357.77,207.90 358.29,208.07 358.80,208.23 359.31,208.40 359.82,208.58 360.33,208.76 360.84,208.93 361.34,209.12 361.85,209.30 362.35,209.49 362.85,209.68 363.35,209.87 363.85,210.06 364.35,210.26 364.84,210.46 365.33,210.66 365.83,210.87 366.32,211.08 366.81,211.29 367.29,211.50 367.78,211.72 368.26,211.94 368.75,212.16 369.23,212.38 369.71,212.61 370.18,212.84 370.66,213.07 371.13,213.30 371.60,213.54 372.08,213.78 372.54,214.02 373.01,214.26 373.48,214.51 373.94,214.76 374.40,215.01 374.86,215.27 375.32,215.52 375.77,215.78 376.23,216.04 376.68,216.31 377.13,216.58 377.57,216.85 378.02,2

### Lecture du résultat — l'issue orbite périodique du Matching Pennies

Deuxième issue : la trajectoire **revient** à son point de départ — le détecteur rend `orbite_periodique`, et la conservation de l'invariant à $\sim 2\cdot10^{-14}$ relative est la preuve numérique que la boucle est bien fermée (chaque orbite vit entière sur une ligne de niveau de $x(1-x)\,y(1-y)$). Aucun des deux joueurs ne converge : l'apprentissage ne s'arrête jamais, il tourne. La linéarisation au centre donne une pulsation $\omega = 1$, donc une période $2\pi \approx 6{,}28$ aux petites amplitudes — vérifions ça, et voyons ce que l'amplitude en fait.

In [5]:
// Les minima et leur espacement suivent la règle du carnet Python.
var deltas = new[] {0.01,0.05,0.15,0.35};
var periodes = new List<double>();
var periodesPythonCitees = new[] {6.284,6.315,6.583,8.555};
Console.WriteLine($"{' '.ToString()+"amplitude",10} {"periode",10} {"2*pi",7}");
for (int k = 0; k < deltas.Length; k++)
{
    double d = deltas[k];
    var (t2,tr2) = Limites.Integrer(rhsMp,Limites.Etat(0.5+d,0.5+d),30.0,1e-3);
    var xs = tr2.Select(z => z[0]).ToArray();
    double seuil = xs.Min()+0.25*d;
    var gardes = new List<int>();
    for (int i = 1; i < xs.Length-1; i++)
        if (xs[i] < xs[i-1] && xs[i] <= xs[i+1] && xs[i] < seuil &&
            (gardes.Count == 0 || t2[i]-t2[gardes[gardes.Count-1]] > 2.0)) gardes.Add(i);
    if (gardes.Count < 2) throw new Exception("Deux minima requis pour mesurer la période");
    double periode = t2[gardes[1]]-t2[gardes[0]];
    periodes.Add(periode);
    Console.WriteLine($"{d,10:F2} {periode,10:F3} {2*Math.PI,7:F3}");
    Limites.Verifier($"Période amplitude {d:F2}",periode,periodesPythonCitees[k],0.0011);
}
// SVG calculé depuis la table des périodes et la référence linéaire 2π.
var graphiquePeriodes = new StringBuilder();
graphiquePeriodes.Append("<svg xmlns='http://www.w3.org/2000/svg' width='700' height='450' viewBox='0 0 700 450'><rect width='700' height='450' fill='white'/><rect x='70' y='45' width='560' height='330' fill='none' stroke='black'/><text x='350' y='24' text-anchor='middle'>Matching Pennies : période et amplitude</text><text x='350' y='425' text-anchor='middle'>amplitude initiale</text><text x='18' y='210' transform='rotate(-90 18 210)' text-anchor='middle'>période</text>");
Func<double,double> pxPer = x => 70 + 560*x/0.4;
Func<double,double> pyPer = y => 375 - 330*(y-6.0)/3.0;
graphiquePeriodes.AppendFormat(CultureInfo.InvariantCulture,"<line x1='70' y1='{0:F2}' x2='630' y2='{0:F2}' stroke='black' stroke-dasharray='6,4'/><text x='475' y='{1:F2}'>2π (linéaire)</text>",pyPer(2*Math.PI),pyPer(2*Math.PI)-6);
graphiquePeriodes.Append("<polyline fill='none' stroke='#9467bd' points='");
for (int i = 0; i < deltas.Length; i++) graphiquePeriodes.AppendFormat(CultureInfo.InvariantCulture,"{0:F2},{1:F2} ",pxPer(deltas[i]),pyPer(periodes[i]));
graphiquePeriodes.Append("'/>");
for (int i = 0; i < deltas.Length; i++) graphiquePeriodes.AppendFormat(CultureInfo.InvariantCulture,"<circle cx='{0:F2}' cy='{1:F2}' r='4' fill='#9467bd'/><text x='{0:F2}' y='395' text-anchor='middle' font-size='11'>{2:F2}</text>",pxPer(deltas[i]),pyPer(periodes[i]),deltas[i]);
graphiquePeriodes.Append("</svg>");
display(graphiquePeriodes.ToString(),"text/html");

 amplitude    periode    2*pi


      0,01      6,284   6,283


Période amplitude 0,01 : calculé 6,284000E+000, référence Python CITÉE 6,284000E+000, écart 0,000E+000, tolérance 1,100E-003


      0,05      6,315   6,283


Période amplitude 0,05 : calculé 6,315000E+000, référence Python CITÉE 6,315000E+000, écart 0,000E+000, tolérance 1,100E-003


      0,15      6,583   6,283


Période amplitude 0,15 : calculé 6,583000E+000, référence Python CITÉE 6,583000E+000, écart 8,882E-016, tolérance 1,100E-003


      0,35      8,555   6,283


Période amplitude 0,35 : calculé 8,555000E+000, référence Python CITÉE 8,555000E+000, écart 0,000E+000, tolérance 1,100E-003


Matching Pennies : période et amplitude amplitude initiale période 2π (linéaire) 0.01 0.05 0.15 0.35

### Lecture du résultat — la période dérive avec l'amplitude

Aux petites amplitudes, la période mesurée colle à la prédiction $2\pi$ de la linéarisation — puis elle **croît** avec l'amplitude : l'orbite est fermée mais le champ la ralentit loin du centre. Le tableau le chiffre : amplitude 0,01 -> période 6,284 contre 2pi = 6,283, écart +0,001 (0,016 %) ; 0,05 -> 6,315, écart +0,032 (0,5 %) ; 0,15 -> 6,583, écart +0,300 (4,8 %) ; 0,35 -> 8,555, écart +2,272 (36 %). Quand l'amplitude est multipliée par 35 (0,01 -> 0,35), l'écart absolu est multiplié par plus de 2000 : à 0,01 du centre la linéarisation est exacte au millième près, à 0,35 elle sous-estime la période d'un tiers.

La cause est dans le champ : les orbites de Matching Pennies vivent sur les lignes de niveau de l'invariant $x(1-x)y(1-y)$, et le champ y est plus lent loin du centre — même forme de boucle, temps de parcours plus long. La linéarisation décrit le voisinage du centre, pas l'orbite entière ; en pratique, la pulsation $\omega = 1$ est une borne de basse amplitude, et toute mesure de période faite au-delà de ~0,1 d'amplitude doit s'attendre à la voir glisser.

### 2.3 Pierre-Feuille-Ciseaux, $w < l$ — l'issue cycle hétéroclinique

Une population, trois stratégies. La victoire rapporte $w$, la défauche coûte $-l$. Quand perdre coûte plus que gagner ne rapporte ($w < l$), le barycentre devient instable et la trajectoire, partie de l'intérieur, s'enroule vers la frontière — sans jamais s'y arrêter : elle colle à chaque coin (Pierre élimine Ciseaux, Feuille enveloppe Pierre, Ciseaux coupent Feuille) puis le quitte, de plus en plus lentement.

In [6]:
// RK4 et détecteur porté : les séjours sont les écarts entre changements de dominance.
var aHet = Limites.Rps(1.0,2.0);
var (tHet,trHet) = Limites.Integrer(z => Limites.Champ1(aHet,z),new[] {0.4,0.35,0.25},120.0,1e-3);
var (regimeHet,infoHet) = Limites.Detecter(tHet,trHet);
var sejours = infoHet.Sejours;
Console.WriteLine($"verdict : {regimeHet}");
Console.WriteLine("temps de sejour par sommet dominant :");
Console.WriteLine("  " + string.Join("  ",sejours.Select(g => g.ToString("F1").PadLeft(6))));
if (regimeHet != "cycle_heteroclinique") throw new Exception("Verdict différent de la référence Python CITÉE");
var sejoursPythonCites = new[] {3.7,2.0,2.5,2.9,3.8,5.4,8.8,15.5,28.9};
if (sejours.Length != sejoursPythonCites.Length) throw new Exception("Nombre de séjours différent de la référence Python CITÉE");
double rapportHet = sejours[sejours.Length-1]/Limites.Mediane(sejours.Take(sejours.Length-1));
Console.WriteLine($"rapport dernier / mediane des precedents : {rapportHet:F1}");
Console.WriteLine($"coord minimale de la fenetre terminale  : {infoHet.DistanceFrontiere:E2}");
// Références citées telles qu'arrondies dans la sortie Python.
for (int i = 0; i < sejours.Length; i++) Limites.Verifier($"Séjour {i+1}",sejours[i],sejoursPythonCites[i],0.051);
Limites.Verifier("Rapport des séjours",rapportHet,7.7,0.051);
Limites.Verifier("Distance à la frontière",infoHet.DistanceFrontiere,6.02e-41,0.05e-41);
// Portrait du simplexe et trajectoire calculée.
var svgHet = new StringBuilder();
svgHet.Append("<svg xmlns='http://www.w3.org/2000/svg' width='650' height='600' viewBox='0 0 650 600'><rect width='650' height='600' fill='white'/><text x='325' y='25' text-anchor='middle'>RPS w=1 &lt; l=2 : spirale vers la frontière</text><polygon points='65,535 585,535 65,45' fill='none' stroke='black'/><text x='585' y='555'>Pierre</text><text x='65' y='38'>Feuille</text><text x='65' y='555'>Ciseaux</text><text x='325' y='580' text-anchor='middle'>x_Pierre</text><text x='15' y='290' transform='rotate(-90 15 290)' text-anchor='middle'>x_Feuille</text><polyline fill='none' stroke='#ff7f0e' stroke-width='0.7' points='");
for (int i = 0; i < trHet.Length; i += 20) svgHet.AppendFormat(CultureInfo.InvariantCulture,"{0:F3},{1:F3} ",65+520*trHet[i][0],535-490*trHet[i][1]);
svgHet.AppendFormat(CultureInfo.InvariantCulture,"{0:F3},{1:F3}'/></svg>",65+520*trHet[trHet.Length-1][0],535-490*trHet[trHet.Length-1][1]);
display(svgHet.ToString(),"text/html");

verdict : cycle_heteroclinique


temps de sejour par sommet dominant :


     3,7     2,0     2,5     2,9     3,8     5,4     8,8    15,5    28,9


rapport dernier / mediane des precedents : 7,7


coord minimale de la fenetre terminale  : 6,02E-041


Séjour 1 : calculé 3,739000E+000, référence Python CITÉE 3,700000E+000, écart 3,900E-002, tolérance 5,100E-002


... (7 outputs left out)

Séjour 9 : calculé 2,888700E+001, référence Python CITÉE 2,890000E+001, écart 1,300E-002, tolérance 5,100E-002


Rapport des séjours : calculé 7,709367E+000, référence Python CITÉE 7,700000E+000, écart 9,367E-003, tolérance 5,100E-002


Distance à la frontière : calculé 6,024273E-041, référence Python CITÉE 6,020000E-041, écart 4,273E-044, tolérance 5,000E-043


RPS w=1 < l=2 : spirale vers la frontière Pierre Feuille Ciseaux x_Pierre x_Feuille <polyline fill='none' stroke='#ff7f0e' stroke-width='0.7' points='273.000,363.500 272.483,362.718 271.952,361.931 271.406,361.140 270.846,360.346 270.272,359.548 269.684,358.747 269.082,357.943 268.467,357.136 267.838,356.327 267.196,355.515 266.541,354.701 265.873,353.886 265.193,353.068 264.499,352.250 263.794,351.430 263.076,350.610 262.347,349.789 261.605,348.968 260.853,348.147 260.089,347.326 259.314,346.506 258.529,345.687 257.733,344.869 256.927,344.053 256.111,343.238 255.286,342.426 254.451,341.617 253.606,340.810 252.753,340.006 251.892,339.205 251.022,338.409 250.144,337.616 249.259,336.828 248.366,336.045 247.466,335.267 246.559,334.494 245.646,333.727 244.727,332.967 243.801,332.212 242.871,331.465 241.935,330.725 240.994,329.992 240.048,329.267 239.099,328.550 238.145,327.842 237.188,327.143 236.227,326.454 235.264,325.773 234.298,325.103 233.330,324.444 232.359,323.794 231.387,323.156 230.414,322.530 229.440,321.915 228.465,321.312 227.489,320.721 226.514,320.144 225.539,319.579 224.565,319.028 223.591,318.491 222.619,317.967 221.648,317.458 220.680,316.964 219.713,316.485 218.749,316.021 217.788,315.573 216.830,315.141 215.875,314.726 214.924,314.326 213.977,313.944 213.035,313.579 212.096,313.231 211.163,312.902 210.235,312.590 209.312,312.296 208.395,312.021 207.484,311.764 206.579,311.527 205.681,311.309 204.789,311.110 203.905,310.931 203.027,310.772 202.157,310.633 201.295,310.514 200.440,310.416 199.594,310.339 198.756,310.282 197.926,310.246 197.106,310.232 196.294,310.239 195.492,310.267 194.699,310.317 193.915,310.389 193.141,310.483 192.378,310.598 191.624,310.736 190.880,310.896 190.148,311.077 189.425,311.282 188.714,311.508 188.013,311.757 187.324,312.029 186.646,312.322 185.979,312.639 185.323,312.977 184.680,313.338 184.048,313.722 183.428,314.128 182.820,314.556 182.224,315.007 181.640,315.480 181.069,315.975 180.511,316.493 179.964,317.032 179.431,317.594 178.910,318.177 178.403,318.782 177.908,319.408 177.426,320.056 176.957,320.726 176.502,321.416 176.060,322.127 175.631,322.859 175.216,323.612 174.815,324.385 174.427,325.178 174.052,325.991 173.692,326.824 173.345,327.676 173.012,328.547 172.693,329.438 172.388,330.347 172.098,331.274 171.821,332.219 171.559,333.182 171.311,334.162 171.077,335.160 170.857,336.174 170.652,337.204 170.462,338.251 170.286,339.313 170.125,340.390 169.978,341.483 169.846,342.590 169.729,343.711 169.627,344.845 169.540,345.993 169.467,347.154 169.410,348.328 169.367,349.513 169.340,350.710 169.328,351.918 169.332,353.136 169.350,354.365 169.384,355.604 169.433,356.852 169.498,358.108 169.579,359.373 169.675,360.646 169.786,361.927 169.914,363.214 170.057,364.507 170.216,365.806 170.391,367.111 170.582,368.421 170.789,369.735 171.012,371.053 171.252,372.374 171.508,373.698 171.780,375.025 172.068,376.354 172.373,377.684 172.695,379.015 173.033,380.346 173.388,381.677 173.760,383.008 174.148,384.338 174.554,385.666 174.977,386.993 175.416,388.317 175.873,389.638 176.348,390.956 176.839,392.270 177.348,393.579 177.875,394.885 178.419,396.185 178.981,397.479 179.560,398.768 180.158,400.050 180.773,401.325 181.407,402.593 182.058,403.854 182.728,405.107 183.416,406.352 184.122,407.587 184.847,408.814 185.590,410.032 186.351,411.240 187.132,412.438 187.931,413.625 188.749,414.802 189.585,415.968 190.441,417.123 191.315,418.267 192.209,419.399 193.121,420.518 194.053,421.626 195.004,422.721 195.974,423.803 196.963,424.873 197.972,425.929 199.000,426.972 200.048,428.002 201.115,429.018 202.201,430.020 203.307,431.008 204.432,431.981 205.577,432.941 206.742,433.886 207.925,434.817 209.129,435.732 210.351,436.633 211.594,437.520 212.855,438.391 214.136,439.247 215.436,440.088 216.756,440.913 218.094,441.723 219.452,442.518 220.829,443.298 222.225,444.062 223.639,444.810 225.072,445.543 226.524,446.260 227.994,446.962 229.483,447.648 230.989,448.318 232.514,448.973 234.056,449.612 235.616,45

### Lecture du résultat — le ralentissement hétéroclinique, auto-accéléré

Troisième issue : ni convergence, ni retour. La trajectoire s'approche du cycle Pierre -> Feuille -> Ciseaux -> Pierre qui relie les trois coins de la frontière — un **cycle hétéroclinique** — et les temps de séjour par sommet s'allongent au fil du run : la fenêtre rend neuf séjours dominants 3,7 - 2,0 - 2,5 - 2,9 - 3,8 - 5,4 - 8,8 - 15,5 - 28,9. Les quatre premiers se tiennent entre 2,0 et 3,8 (le run démarre encore loin des coins) ; à partir du cinquième, chaque séjour vaut environ une fois et demie le précédent : 3,8 -> 5,4 (x1,42), 5,4 -> 8,8 (x1,63), 8,8 -> 15,5 (x1,76), 15,5 -> 28,9 (x1,86) — et le facteur lui-même croît, vers x2. C'est la signature d'un ralentissement hétéroclinique auto-accéléré : plus la trajectoire colle à la frontière, plus le champ y est lent, plus elle s'y colle.

La coordonnée minimale de la fenêtre terminale, 6,02e-41, dit la même chose autrement : à 10^-41 du bord, la trajectoire n'a toujours atteint aucun sommet — l'ensemble omega-limite est le cycle tout entier, jamais un coin isolé. (Un test de régression du module couvre le cas extrême : à $t_{\max} = 200$, la fenêtre terminale semble immobile alors que la trajectoire est simplement en pleine rampe près d'un coin — chaque départ de dominance constaté réfute la convergence.)

## 3. Le mur $w = l$ : une paroi dans l'espace des jeux

Le [GameTheory-3b](GameTheory-03b-Chambres-Murs-Chemins-Lean-Python.ipynb) parcourt l'espace des jeux comme un arrangement de **chambres** séparées par des **murs** — les égalités. La famille RPS paramétrée par $(w, l)$ traverse exactement un tel mur : la diagonale $w = l$. De part et d'autre, deux chambres de régimes ; sur le mur, le troisième type.

La linéarisation au barycentre le dit en une ligne : sur le plan tangent, les valeurs propres du champ ont pour partie réelle $(l - w)/6$ — négative si $w > l$ (barycentre attracteur, spirale rentrante), positive si $w < l$ (barycentre répulsif, $\omega$-limite rejetée vers la frontière), nulle sur le mur (centre). Vérifions la formule numériquement, puis balayons le mur.

In [7]:
// Différences finies centrées sur les trois coordonnées ambiantes.
double[,] JacobienComplet(double[,] a, double eps = 1e-6)
{
    var j = new double[3,3];
    for (int k = 0; k < 3; k++)
    {
        var plus = new[] {1.0/3,1.0/3,1.0/3};
        var moins = (double[])plus.Clone();
        plus[k] += eps;
        moins[k] -= eps;
        var fp = Limites.Champ1(a,plus);
        var fm = Limites.Champ1(a,moins);
        for (int i = 0; i < 3; i++) j[i,k] = (fp[i]-fm[i])/(2*eps);
    }
    return j;
}

// Les formules ci-dessous sont des attentes analytiques, et non des sorties Python.
void VerifierFormule(string nom, double valeur, double attendu, double tolerance)
{
    double ecart = Math.Abs(valeur-attendu);
    Console.WriteLine($"{nom} : calculé {valeur:E6}, formule analytique {attendu:E6}, écart {ecart:E3}, tolérance {tolerance:E3}");
    if (double.IsNaN(valeur) || ecart > tolerance) throw new Exception($"Écart : {nom}");
}

// Le jacobien 3x3 porte une valeur propre radiale en plus de la paire tangente.
Console.WriteLine($"{"w",4} {"l",4} {"vp tangente",20} {"(l-w)/6",9}");
foreach (var cfg in new[] {(w:2.0,l:1.0),(w:1.5,l:1.5),(w:1.0,l:2.0)})
{
    var j = JacobienComplet(Limites.Rps(cfg.w,cfg.l));
    double trace = j[0,0]+j[1,1]+j[2,2];
    double c2 = -trace;
    double c1 = j[0,0]*j[1,1]+j[0,0]*j[2,2]+j[1,1]*j[2,2]
        -j[0,1]*j[1,0]-j[0,2]*j[2,0]-j[1,2]*j[2,1];
    double det = j[0,0]*(j[1,1]*j[2,2]-j[1,2]*j[2,1])
        -j[0,1]*(j[1,0]*j[2,2]-j[1,2]*j[2,0])
        +j[0,2]*(j[1,0]*j[2,1]-j[1,1]*j[2,0]);
    double c0 = -det;
    double radial = (cfg.l-cfg.w)/3.0;
    double reel = (cfg.l-cfg.w)/6.0;
    double imag = (cfg.w+cfg.l)/(2.0*Math.Sqrt(3.0));
    double q = reel*reel+imag*imag;
    VerifierFormule($"J {cfg.w:F1},{cfg.l:F1} coefficient λ²",c2,-(radial+2*reel),1e-8);
    VerifierFormule($"J {cfg.w:F1},{cfg.l:F1} coefficient λ",c1,q+2*radial*reel,1e-8);
    VerifierFormule($"J {cfg.w:F1},{cfg.l:F1} terme constant",c0,-radial*q,1e-8);
    double radialMesure = 0;
    for (int i = 0; i < 3; i++) for (int k = 0; k < 3; k++) radialMesure += j[i,k]/3.0;
    VerifierFormule("Troisième valeur propre",radialMesure,radial,1e-8);
    double reelMesure = (trace-radialMesure)/2;
    double imagCarre = c1-2*radialMesure*reelMesure-reelMesure*reelMesure;
    if (imagCarre < 0) throw new Exception("Paire tangente non complexe");
    double imagMesure = Math.Sqrt(imagCarre);
    VerifierFormule("Partie réelle tangente",reelMesure,reel,5e-3);
    VerifierFormule("Partie imaginaire tangente",imagMesure,imag,1e-8);
    // Valeurs tangentielles citées de la sortie Python, arrondies à quatre décimales.
    double reelPythonCite = cfg.w > cfg.l ? -0.1667 : cfg.w < cfg.l ? 0.1667 : 0.0;
    Limites.Verifier("Partie réelle tangente, sortie Python",reelMesure,reelPythonCite,0.00005);
    Limites.Verifier("Partie imaginaire tangente, sortie Python",imagMesure,0.8660,0.00005);
    Console.WriteLine($"{cfg.w,4:F1} {cfg.l,4:F1} {reelMesure,+12:0.0000;-0.0000;+0.0000}{imagMesure:+0.0000;-0.0000;+0.0000}i {reel,+9:0.0000;-0.0000;+0.0000}");
    Console.WriteLine($"Troisième valeur propre calculée : {radialMesure:+0.000000;-0.000000;+0.000000}");
}

   w    l          vp tangente   (l-w)/6


J 2,0,1,0 coefficient λ² : calculé 6,666667E-001, formule analytique 6,666667E-001, écart 1,784E-011, tolérance 1,000E-008


J 2,0,1,0 coefficient λ : calculé 8,888889E-001, formule analytique 8,888889E-001, écart 4,757E-011, tolérance 1,000E-008


J 2,0,1,0 terme constant : calculé 2,592593E-001, formule analytique 2,592593E-001, écart 2,081E-011, tolérance 1,000E-008


Troisième valeur propre : calculé -3,333333E-001, formule analytique -3,333333E-001, écart 8,919E-012, tolérance 1,000E-008


Partie réelle tangente : calculé -1,666667E-001, formule analytique -1,666667E-001, écart 4,459E-012, tolérance 5,000E-003


... (21 outputs left out)

Partie réelle tangente, sortie Python : calculé 1,666667E-001, référence Python CITÉE 1,667000E-001, écart 3,333E-005, tolérance 5,000E-005


Partie imaginaire tangente, sortie Python : calculé 8,660254E-001, référence Python CITÉE 8,660000E-001, écart 2,540E-005, tolérance 5,000E-005


 1,0  2,0       0,1667+0,8660i    0,1667


Troisième valeur propre calculée : +0,333333


In [8]:
// Les cinq verdicts et les valeurs affichées sont comparés aux sorties citées du jumeau Python.
var configsMur = new[] {(w:2.5,l:0.5),(w:2.0,l:1.0),(w:1.5,l:1.5),(w:1.0,l:2.0),(w:0.5,l:2.5)};
var verdictsPythonCites = new[] {"point_fixe","point_fixe","orbite_periodique","cycle_heteroclinique","cycle_heteroclinique"};
// Valeurs de (l-w)/6 citées avec les trois décimales affichées par Python.
var reelsPythonCites = new[] {-0.333,-0.167,0.000,0.167,0.333};
Console.WriteLine($"{"w",4} {"l",4} {"(l-w)/6",9}  {"regime mesure",-20}");
for (int i = 0; i < configsMur.Length; i++)
{
    var cfg = configsMur[i];
    var a = Limites.Rps(cfg.w,cfg.l);
    var (temps,trajet) = Limites.Integrer(z => Limites.Champ1(a,z),new[] {0.4,0.35,0.25},120.0,1e-3);
    var (verdict,info) = Limites.Detecter(temps,trajet);
    double reel = (cfg.l-cfg.w)/6;
    Console.WriteLine($"{cfg.w,4:F1} {cfg.l,4:F1} {reel,+9:0.000;-0.000;+0.000}  {verdict,-20}");
    Console.WriteLine($"Référence Python CITÉE : {verdictsPythonCites[i]}");
    if (verdict != verdictsPythonCites[i]) throw new Exception($"Verdict différent pour w={cfg.w}, l={cfg.l}");
    Limites.Verifier($"(l-w)/6 pour w={cfg.w:F1}, l={cfg.l:F1}",reel,reelsPythonCites[i],0.0005);
}

// Trois portraits à l'horizon 80, avec les mêmes conditions initiales.
var portraitsMur = new StringBuilder();
portraitsMur.Append("<svg xmlns='http://www.w3.org/2000/svg' width='1500' height='480' viewBox='0 0 1500 480'><rect width='1500' height='480' fill='white'/>");
var configsPortrait = new[] {(w:2.0,l:1.0),(w:1.5,l:1.5),(w:1.0,l:2.0)};
for (int p = 0; p < configsPortrait.Length; p++)
{
    var cfg = configsPortrait[p];
    var a = Limites.Rps(cfg.w,cfg.l);
    var (temps,trajet) = Limites.Integrer(z => Limites.Champ1(a,z),new[] {0.4,0.35,0.25},80.0,1e-3);
    double x0 = 50+500*p;
    string nom = cfg.w == cfg.l ? "mur w=l" : cfg.w < cfg.l ? "w &lt; l" : "w &gt; l";
    portraitsMur.AppendFormat(CultureInfo.InvariantCulture,"<text x='{0:F1}' y='25' text-anchor='middle'>{1} (w={2:F1}, l={3:F1}) : partie réelle {4:+0.00;-0.00;+0.00}</text>",x0+190,nom,cfg.w,cfg.l,(cfg.l-cfg.w)/6);
    portraitsMur.AppendFormat(CultureInfo.InvariantCulture,"<polygon points='{0:F1},420 {1:F1},420 {0:F1},55' fill='none' stroke='black'/><polyline fill='none' stroke='#1f77b4' stroke-width='0.8' points='",x0,x0+365);
    for (int k = 0; k < trajet.Length; k += 20) portraitsMur.AppendFormat(CultureInfo.InvariantCulture,"{0:F2},{1:F2} ",x0+365*trajet[k][0],420-365*trajet[k][1]);
    portraitsMur.AppendFormat(CultureInfo.InvariantCulture,"{0:F2},{1:F2}'/><text x='{2:F1}' y='450' text-anchor='middle'>x_Pierre</text>",x0+365*trajet[trajet.Length-1][0],420-365*trajet[trajet.Length-1][1],x0+180);
}
portraitsMur.Append("<text x='15' y='230' transform='rotate(-90 15 230)' text-anchor='middle'>x_Feuille</text></svg>");
display(portraitsMur.ToString(),"text/html");

// Bisection du seuil de classification à horizon fini, pas du mur analytique.
string VerdictPourW(double w)
{
    var a = Limites.Rps(w,2.0);
    var (temps,trajet) = Limites.Integrer(z => Limites.Champ1(a,z),new[] {0.4,0.35,0.25},120.0,1e-3);
    return Limites.Detecter(temps,trajet).verdict;
}
double borneAutre = 1.0, borneFixe = 3.0;
if (VerdictPourW(borneAutre) == "point_fixe" || VerdictPourW(borneFixe) != "point_fixe") throw new Exception("Bornes de classification invalides");
while (borneFixe-borneAutre > 0.01)
{
    double milieu = (borneAutre+borneFixe)/2;
    if (VerdictPourW(milieu) == "point_fixe") borneFixe = milieu;
    else borneAutre = milieu;
}
Console.WriteLine($"Seuil de classification à horizon fini : [{borneAutre:F4}, {borneFixe:F4}] ; largeur {borneFixe-borneAutre:F4}");
Console.WriteLine($"Mur analytique w=2 : écart du milieu {Math.Abs((borneAutre+borneFixe)/2-2.0):F4}");
if (borneFixe-borneAutre > 0.01) throw new Exception("Intervalle de bisection trop large");

   w    l   (l-w)/6  regime mesure       


 2,5  0,5    -0,333  point_fixe          


Référence Python CITÉE : point_fixe


(l-w)/6 pour w=2,5, l=0,5 : calculé -3,333333E-001, référence Python CITÉE -3,330000E-001, écart 3,333E-004, tolérance 5,000E-004


 2,0  1,0    -0,167  point_fixe          


Référence Python CITÉE : point_fixe


... (9 outputs left out)

(l-w)/6 pour w=0,5, l=2,5 : calculé 3,333333E-001, référence Python CITÉE 3,330000E-001, écart 3,333E-004, tolérance 5,000E-004


w > l (w=2.0, l=1.0) : partie réelle -0.17 <polyline fill='none' stroke='#1f77b4' stroke-width='0.8' points='196.00,292.25 195.48,291.68 194.96,291.12 194.43,290.57 193.90,290.01 193.37,289.47 192.83,288.93 192.29,288.39 191.75,287.86 191.21,287.34 190.67,286.82 190.13,286.31 189.58,285.81 189.03,285.31 188.48,284.82 187.93,284.33 187.39,283.85 186.84,283.38 186.29,282.92 185.74,282.47 185.19,282.02 184.64,281.58 184.09,281.15 183.54,280.72 183.00,280.31 182.45,279.90 181.91,279.50 181.36,279.11 180.82,278.73 180.28,278.35 179.75,277.99 179.21,277.63 178.68,277.28 178.15,276.95 177.62,276.62 177.09,276.30 176.57,275.99 176.05,275.68 175.53,275.39 175.02,275.11 174.51,274.84 174.01,274.57 173.50,274.32 173.00,274.07 172.51,273.84 172.02,273.61 171.53,273.40 171.05,273.19 170.57,273.00 170.09,272.81 169.62,272.63 169.16,272.47 168.70,272.31 168.24,272.16 167.79,272.02 167.35,271.90 166.90,271.78 166.47,271.67 166.04,271.57 165.61,271.48 165.19,271.40 164.78,271.34 164.37,271.28 163.96,271.23 163.56,271.18 163.17,271.15 162.78,271.13 162.40,271.12 162.02,271.11 161.65,271.12 161.29,271.14 160.93,271.16 160.58,271.19 160.23,271.23 159.89,271.29 159.56,271.34 159.23,271.41 158.90,271.49 158.59,271.57 158.28,271.67 157.97,271.77 157.68,271.88 157.38,272.00 157.10,272.13 156.82,272.26 156.55,272.40 156.28,272.55 156.02,272.71 155.77,272.87 155.52,273.04 155.28,273.22 155.04,273.41 154.82,273.60 154.59,273.80 154.38,274.01 154.17,274.22 153.97,274.44 153.77,274.67 153.58,274.90 153.40,275.14 153.22,275.38 153.05,275.63 152.89,275.89 152.73,276.15 152.58,276.42 152.43,276.69 152.29,276.96 152.16,277.25 152.03,277.53 151.91,277.82 151.80,278.12 151.69,278.42 151.59,278.72 151.50,279.03 151.41,279.34 151.33,279.66 151.25,279.98 151.18,280.30 151.12,280.63 151.06,280.96 151.01,281.29 150.96,281.62 150.92,281.96 150.89,282.30 150.86,282.65 150.84,282.99 150.82,283.34 150.81,283.69 150.81,284.04 150.81,284.39 150.82,284.75 150.83,285.11 150.85,285.47 150.87,285.82 150.90,286.19 150.94,286.55 150.98,286.91 151.03,287.27 151.08,287.64 151.14,288.00 151.20,288.37 151.27,288.73 151.34,289.09 151.42,289.46 151.51,289.82 151.60,290.19 151.69,290.55 151.79,290.92 151.90,291.28 152.01,291.64 152.12,292.00 152.24,292.36 152.36,292.72 152.49,293.08 152.63,293.44 152.76,293.80 152.91,294.15 153.06,294.50 153.21,294.86 153.36,295.20 153.52,295.55 153.69,295.90 153.86,296.24 154.03,296.58 154.21,296.92 154.39,297.26 154.58,297.60 154.77,297.93 154.96,298.26 155.16,298.59 155.36,298.91 155.56,299.23 155.77,299.55 155.98,299.87 156.20,300.18 156.42,300.49 156.64,300.80 156.86,301.10 157.09,301.40 157.32,301.70 157.55,302.00 157.79,302.29 158.03,302.58 158.27,302.86 158.52,303.14 158.77,303.42 159.02,303.69 159.27,303.96 159.52,304.23 159.78,304.49 160.04,304.75 160.30,305.00 160.56,305.25 160.83,305.50 161.10,305.74 161.37,305.98 161.64,306.21 161.91,306.45 162.18,306.67 162.46,306.89 162.73,307.11 163.01,307.33 163.29,307.54 163.57,307.74 163.85,307.94 164.13,308.14 164.42,308.33 164.70,308.52 164.98,308.70 165.27,308.88 165.55,309.06 165.84,309.23 166.12,309.40 166.41,309.56 166.69,309.72 166.98,309.87 167.26,310.02 167.55,310.16 167.83,310.30 168.12,310.44 168.40,310.57 168.69,310.70 168.97,310.82 169.25,310.94 169.53,311.05 169.82,311.16 170.10,311.26 170.37,311.36 170.65,311.46 170.93,311.55 171.20,311.63 171.48,311.72 171.75,311.79 172.02,311.87 172.29,311.94 172.56,312.00 172.82,312.06 173.09,312.12 173.35,312.17 173.61,312.21 173.87,312.26 174.12,312.30 174.38,312.33 174.63,312.36 174.88,312.39 175.12,312.41 175.37,312.42 175.61,312.44 175.85,312.45 176.08,312.45 176.31,312.45 176.55,312.45 176.77,312.44 177.00,312.43 177.22,312.41 177.44,312.39 177.65,312.37 177.86,312.34 178.07,312.31 178.28,312.28 178.48,312.24 178.68,312.19 178.87,312.15 179.07,312.10 179.25,312.04 179.44,311.98 179.62,311.92 179.80,311.86 179.97,311.79 180.14,311.72 180.31,311.64 180.47,311.56 180.63,311.48 180.78,311.40 180.93,311.31 181.08,311.21 181.22,311.12 181.36,311.02

Seuil de classification à horizon fini : [2,2656, 2,2734] ; largeur 0,0078


Mur analytique w=2 : écart du milieu 0,2695


### Lecture du résultat — le mur ne change que le sens radial

Le signe de $(l-w)/6$ **prédit** le régime mesuré sur toute la ligne $w + l = 3$ : négatif à droite du mur — spirale rentrante vers le barycentre, `point_fixe` ; positif à gauche — rejet vers le cycle de la frontière, `cycle_heteroclinique` ; nul **sur** le mur — orbites neutres fermées, `orbite_periodique`. Les deux tableaux se répondent ligne à ligne : le jacobien au barycentre porte $w=2, l=1$ -> $-0{,}1667 + 0{,}8660i$ ; mur $w=l=1{,}5$ -> $0{,}0000 + 0{,}8660i$ ; $w=1, l=2$ -> $+0{,}1667 + 0{,}8660i$ — la partie réelle vaut exactement $(l-w)/6$ ($-1/6$, $0$, $+1/6$), et le tableau de mesures l'entérine sur cinq lignes : $-0{,}333$ et $-0{,}167$ -> `point_fixe`, $0{,}000$ -> `orbite_periodique`, $+0{,}167$ et $+0{,}333$ -> `cycle_heteroclinique`.

Mais la partie imaginaire reste $0{,}8660 = \sqrt{3}/2$ sur toute la ligne : la pulsation de rotation autour du barycentre ne dépend pas de la position le long de la ligne — traverser le mur change le sens radial (rentrant, neutre, sortant), jamais la vitesse angulaire. Les trois portraits disent la même chose géométriquement : en traversant le mur $w = l$, l'$\omega$-limite saute du barycentre aux orbites fermées, puis au cycle de la frontière. Un paramètre du jeu déplace l'équilibre entre les deux axes du plan, pas la géométrie de la rotation. C'est exactement le vocabulaire du 3b : une paroi de codimension 1 dans l'espace des familles de jeux, deux chambres de régimes de part et d'autre — et le régime du mur lui-même.

## 4. Exercices

### Exercice 1 — Chasse au Cerf : un jeu, deux bassins

Le module fournit `Limites.ChasseAuCerf()` : $A = \begin{pmatrix} 4 & 1 \\ 3 & 2 \end{pmatrix}$, $B = A^\top$ (la transposée n'est **pas** optionnelle : $A$ n'est pas symétrique). Deux conditions initiales vous sont données. Pour chacune : intégrer, classer le régime, identifier l'attracteur atteint — puis dire ce qui sépare les deux issues.

In [9]:
// Exercice 1 -- Chasse au Cerf : identifier le regime et les bassins
var (aCerf,bCerf) = Limites.ChasseAuCerf();
var departHaut = (0.62,0.62); // x + y > 1
var departBas = (0.30,0.30); // x + y < 1
// Etape 1 : integrer les deux trajectoires (t_max=60, dt=1e-3).
// Etape 2 : pour chacune, appliquer Detecter et lire l'etat final.
// Etape 3 : formuler ce qui separe les deux bassins (indice : une droite du
// carre passe par la selle interieure ; laquelle ?).
object verdictExo1 = null; // TODO etudiant : (regime_haut, regime_bas, separateur)
Console.WriteLine("Exercice a completer");

Exercice a completer


### Exercice 2 — Personne ne quitte son ensemble limite

Le théorème dit que la trajectoire **reste** sur son ensemble limite. Vérifiez-le par la mesure pour Matching Pennies depuis une autre condition initiale $(0.2, 0.75)$ : l'invariant $x(1-x)\,y(1-y)$ doit rester constant le long de la trajectoire — à quelle précision relative ? Et pour RPS $w < l$, la trajectoire intérieure s'approche de la frontière sans jamais la toucher : quelle est la coordonnée minimale atteinte sur tout le run ?

In [10]:
// Exercice 2 -- mesurer la conservation des ensembles limites
// Etape 1 : MP depuis (0.2, 0.75) -- t_max=20, dt=1e-3 : max_t |inv(t) - inv(0)|
// relatif a inv(0).
// Etape 2 : RPS w=1, l=2 depuis (0.4, 0.35, 0.25) -- t_max=120 : min de toutes
// les coordonnees de la trajectoire (interieur strict conserve ?).
// Indice : Limites.InvariantMp(z) donne l'invariant ; Min donne le minimum.
object conservationExo2 = null; // TODO etudiant : (ecart_relatif_mp, coord_min_rps)
Console.WriteLine("Exercice a completer");

Exercice a completer


### Exercice 3 — Trouver le mur à la bisection

La section 3 a montré le mur $w = l$ sur la ligne $w + l = 3$. On va le trouver **sans** la formule : fixez $l = 2$ et balayez $w$ — pour chaque $w$, intégrez (t_max=120, dt=1e-3) et classez. Par bisection sur le verdict (`point_fixe` vs autre chose), localisez le mur à $\pm 0{,}01$ près, puis comparez à la prédiction $w = l = 2$.

In [11]:
// Exercice 3 -- bisection sur w pour localiser le mur (l = 2 fixe)
// Etape 1 : deux bornes de part et d'autre du mur (verifiez leurs verdicts).
// Etape 2 : bisection ~7 iterations pour une precision de 0.01 sur w.
// Etape 3 : comparer la valeur trouvee a l et commenter l'ecart.
object murExo3 = null; // TODO etudiant : w trouve par bisection
Console.WriteLine("Exercice a completer");

Exercice a completer


## Conclusion

Trois jeux, trois issues, et une liste **fermée** : point fixe, orbite périodique, cycle hétéroclinique. C'est le privilège de la dimension 2, et il a un prix symétrique : **le théorème tombe dès qu'on quitte le plan**. Deux populations sur trois stratégies (dimension 4), une population sur quatre stratégies (dimension 3) — l'alternative n'y est plus exhaustive, et des comportements chaotiques sont documentés dans ces cadres (Hofbauer-Sigmund pour le replicateur à assez de stratégies ; Sato-Akiyama-Crutchfield 2005 pour l'apprentissage avec mémoire en jeu à deux). Ce notebook ne généralise pas : l'échec du théorème au-delà du plan **est** la leçon — la classification que vous venez d'exécuter est un théorème de géométrie, pas une propriété de l'apprentissage en général.

**Source** : Czechowski & Piliouras (2021), *Poincaré-Bendixson Limit Sets in Multi-Agent Learning* — bibliothèque locale : `G:/Mon Drive/MyIA/IA/Bibliographie IA/GameTheory/Nash Equilibrium and Regret Minimization/2021 - Czechowski Piliouras - Poincare-Bendixson Limit Sets in Multi-Agent Learning.pdf` (citée, jamais copiée dans le dépôt).

*Companion : [`limit_sets.py`](limit_sets.py) + [`tests/test_limit_sets.py`](tests/test_limit_sets.py) (16 tests).*

## Attestation de parité

Portage C# du jumeau Python *GameTheory-06i — Ensembles limites : Poincaré-Bendixson en dimension 2*, mesuré cellule par cellule contre la source.

| Section | Source | Livrable | Écart mesuré |
|---|---|---|---|
| Carte des cellules | la carte du carnet Python | même carte, dans le même ordre et sous les mêmes titres | aucune cellule omise ; cette attestation est la seule ajoutée, en fin de carnet |
| §1 Le théorème | 3 issues énumérées | mêmes trois issues | aucun calcul numérique |
| §2 Trois trajectoires | RK4, `dt = 1e-3` | même intégrateur, même pas | verdicts mesurés = références citées |
| §3 Le mur $w = l$ | cinq configurations $(w, l)$ | mêmes cinq configurations | verdicts identiques ; $\lvert(l-w)/6\rvert$ écart max $3{,}333 \times 10^{-4}$ pour une tolérance affichée de $5{,}0 \times 10^{-4}$ |
| §4 Exercices 1–3 | énoncés | stubs compilables `TODO etudiant` | aucune solution révélée |
| Conclusion | texte et bibliographie | même texte | aucun calcul |

Deux écarts sont **affichés comme mesures**, non comme défauts. Le premier est l'écart d'arrondi de $(l-w)/6$ : $3{,}333 \times 10^{-4}$ sur trois configurations non exactes, nul sur $(1{,}5\,; 1{,}5)$ — sous la tolérance affichée dans la cellule de comparaison. Le second est le seuil de classification du détecteur à horizon fini, mesuré dans $[2{,}2656\,; 2{,}2734]$ alors que le mur analytique vaut $w = 2$ (écart du milieu $0{,}2695$) : la source n'affiche pas ce seuil analytique comme valeur attendue, et l'écart est précisément ce que le détecteur à horizon fini mesure.

Tous les calculs sont en `double`. Les grandeurs non exactes par nature portent des tolérances déclarées — `tolPoint = 2e-3`, `tolRetour = 5e-2`, `tolBord = 1e-3` — et chaque comparaison passe par `Verifier(...)`, qui lève une exception dès que l'écart dépasse la tolérance.